In [14]:
# Humor Detection using Ollama Models

## Imports

# ```python
import pandas as pd
import ollama
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()
# ```

## Configuration

# ```python
DATA_PATH = "Data/testing/joke_dialect_detection(testing).csv"

LABELS = [
    "Saudi",
    "Yemen",
    "Morocco",
    "Algeria",
    "Libya",
    "Tunisia"
]

MODELS = [
    
    # "llama3.1",
    # "mistral",
    # "gemma3",
    "command-r7b-arabic",
    # "qwen2.5",
    # "deepseek-r1"
]
# ```

## Prompt

# ```python
def build_prompt(text):

    return f"""
You are an expert in Arabic dialect identification.

Task:
Identify the dialect used in the following joke.

Possible dialect labels:
- Saudi
- Yemen
- Morocco
- Algeria
- Libya
- Tunisia

Rules:
- Focus on dialectal vocabulary, expressions, spelling, and grammar.
- Choose exactly one label.
- Return ONLY the label.
- Do not provide explanations.

Joke:
{text}
"""
# ```

## Prediction Function

# ```python
def predict_label(text, model_name):

    try:

        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"
# ```

## Evaluation

# ```python
df = pd.read_csv(DATA_PATH)
df["label"] = df["label"].astype(str)
results = []

for model_name in MODELS:

    preds = []

    for text in tqdm(
        df["text"],
        total=len(df),
        desc=model_name
    ):
        preds.append(
            predict_label(text, model_name)
        )

    acc = accuracy_score(df["label"], preds)

    f1 = f1_score(
        df["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc,4))
    print("Macro F1:", round(f1,4))

    print(
        classification_report(
            df["label"],
            preds,
            digits=4
        )
    )
    print(confusion_matrix(df["label"], preds))
    tmp = df.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_{model_name}_dialect_identification.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "macro_f1",
    ascending=False
)
# ```


command-r7b-arabic:   0%|          | 0/503 [00:00<?, ?it/s]


command-r7b-arabic
Accuracy: 0.2207
Macro F1: 0.1188
              precision    recall  f1-score   support

     Algeria     0.6000    0.0857    0.1500        70
     INVALID     0.0000    0.0000    0.0000         0
       Libya     0.5000    0.0667    0.1176        60
     Morocco     0.3913    0.1286    0.1935        70
       Saudi     0.2253    0.8913    0.3596        92
     Tunisia     0.0000    0.0000    0.0000        19
       Yemen     0.1299    0.1299    0.1299        77
         nan     0.0000    0.0000    0.0000       115

    accuracy                         0.2207       503
   macro avg     0.2308    0.1628    0.1188       503
weighted avg     0.2587    0.2207    0.1475       503

[[ 6  1  2  5 37  5 14  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  4  0 51  0  5  0]
 [ 1  0  1  9 39  5 15  0]
 [ 0  0  0  1 82  0  9  0]
 [ 0  1  0  0 12  0  6  0]
 [ 0  0  0  1 65  1 10  0]
 [ 3  1  1  7 78  7 18  0]]


/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.13/site-packages/sklea

,model,accuracy,macro_f1
0,command-r7b-arabic,0.220676,0.118839


In [7]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Load CSV
df = pd.read_csv("predictions_gemma3_dialect_identification.csv")

# Select columns
y_true = df["label"]
y_pred = df["prediction"]

# Classification report
print(classification_report(y_true, y_pred, digits=4))

# Confusion matrix
print(confusion_matrix(y_true, y_pred))

TypeError: '<' not supported between instances of 'str' and 'float'

In [2]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 242.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/35.3 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 256.0 MB/s  0:00:00


   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [scipy]

   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [scipy]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [scikit-learn]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [scikit-learn]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [scikit-learn]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn]


Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import ollama

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()


DATA_PATH = "Data/testing/joke_dialect_detection(testing).csv"
TRAIN_PATH = "Data/training/joke_dialect_detection(training).csv"

LABELS = [
    "Saudi",
    "Yemen",
    "Morocco",
    "Algeria",
    "Libya",
    "Tunisia"
]

MODELS = [
    # "qwen2.5",
    # "llama3.1",
    # "mistral",
    # "gemma3",
    "command-r7b-arabic",
    # "deepseek-r1"
]

N_SHOTS = 4

df_test = pd.read_csv(DATA_PATH)
df_test.dropna(subset = ['label'], inplace=True)

df_train = pd.read_csv(TRAIN_PATH)

def build_few_shot_examples(train_df, label, n_shots=2):

    subset = train_df[train_df["label"] == label]

    if len(subset) == 0:
        raise ValueError(f"No training samples found for label: {label}")

    sampled = subset.sample(
        min(n_shots, len(subset)),
        random_state=42
    )

    return [
        (row["text"], row["label"])
        for _, row in sampled.iterrows()
    ]

def build_prompt(text, few_shot_examples):

    example_text = "\n\n".join([
        f"""Text:
{ex_text}

Label:
{ex_label}"""
        for ex_text, ex_label in few_shot_examples
    ])

    return f"""
You are an expert in Arabic dialect identification.

Task:
Identify the dialect used in the following joke.

Possible dialect labels:
- Saudi
- Yemen
- Morocco
- Algeria
- Libya
- Tunisia

Rules:
- Focus on dialectal vocabulary, expressions, spelling, and grammar.
- Choose exactly one label.
- Return ONLY the label.
- Do not provide explanations.

Examples:
{example_text}

Now classify this:

Text:
{text}

Label:
"""


def predict_label(text, model_name, few_shot_examples):

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text, few_shot_examples)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"

results = []

for model_name in MODELS:

    preds = []

    for _, row in tqdm(
        df_test.iterrows(),
        total=len(df_test),
        desc=model_name
    ):

        text = row["text"]
        true_label = row["label"]

        # 🔥 CLASS-CONDITIONED FEW-SHOT (oracle)
        few_shot_examples = build_few_shot_examples(
            df_train,
            label=true_label,
            n_shots=N_SHOTS
        )

        preds.append(
            predict_label(text, model_name, few_shot_examples)
        )

    acc = accuracy_score(df_test["label"], preds)

    f1 = f1_score(
        df_test["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(f1, 4))

    print(
        classification_report(
            df_test["label"],
            preds,
            digits=4
        )
    )
    print(confusion_matrix(df_test["label"], preds))
    tmp = df_test.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_class_conditioned_fewshot_{model_name}_dialect_identification.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values("macro_f1", ascending=False)




command-r7b-arabic:   0%|          | 0/388 [00:00<?, ?it/s]


command-r7b-arabic
Accuracy: 0.7732
Macro F1: 0.7666
              precision    recall  f1-score   support

     Algeria     1.0000    0.6857    0.8136        70
       Libya     0.8889    0.6667    0.7619        60
     Morocco     0.7639    0.7857    0.7746        70
       Saudi     0.7426    0.8152    0.7772        92
     Tunisia     0.5429    1.0000    0.7037        19
       Yemen     0.7241    0.8182    0.7683        77

    accuracy                         0.7732       388
   macro avg     0.7771    0.7952    0.7666       388
weighted avg     0.8020    0.7732    0.7756       388

[[48  1  1  5  6  9]
 [ 0 40  6  8  1  5]
 [ 0  0 55  7  5  3]
 [ 0  0  6 75  4  7]
 [ 0  0  0  0 19  0]
 [ 0  4  4  6  0 63]]


,model,accuracy,macro_f1
0,command-r7b-arabic,0.773196,0.766552
